# SouthWatts case study: a tracking stack you can trust
Independent case study. **PUBLIC DATA**: Google's obfuscated GA4 sample (Google Merchandise Store, Nov 2020 - Jan 2021). **SIMULATED**: 12 weeks of a fictional multinational, Client M (8 markets, 5 platforms). All tables below are read from `data/out`, produced by `src/run_sql.py`, the validators and `src/monitoring.py`.

In [1]:
import json, pandas as pd
from pathlib import Path
OUT = Path('..') / 'data' / 'out'
pd.set_option('display.width', 140); pd.set_option('display.max_columns', 12)
k = json.loads((OUT / 'key_numbers.json').read_text())
csv = lambda n: pd.read_csv(OUT / f'{n}.csv')

## 1. Real audit of the GA4 sample (PUBLIC DATA)
Queries: `sql/01_ga4_real_audit.sql`, `sql/02_ga4_sessions_channels.sql`.

In [2]:
csv('real_audit_summary')

,metric,value
0,purchase_events,5692.0
1,distinct_transaction_ids,4451.0
2,purchases_without_id,906.0
3,duplicated_ids,326.0
4,extra_purchase_rows,335.0
5,page_views,1350428.0
6,page_views_self_referral,206005.0
7,page_views_missing_medium,866938.0
8,page_views_medium_data_deleted,8811.0
9,sessions,359817.0


In [3]:
csv('real_audit_revenue_impact')

,revenue_all_purchase_rows,revenue_rows_without_id,revenue_counted_twice,pct_revenue_counted_twice
0,362165.0,31817.0,22577.33,6.23


In [4]:
csv('real_channels').head(6)

,channel,sessions,purchases,pct_sessions,purchase_rate_pct
0,No source on any event,129747,33.0,36.06,0.025
1,Organic Search,96692,1484.0,26.87,1.535
2,Direct,35318,281.0,9.82,0.796
3,Self-referral (bug),35065,1325.0,9.75,3.779
4,Referral,25536,960.0,7.10,3.759
5,Obfuscated (<Other>),24665,117.0,6.85,0.474


Reading: 906 purchases cannot be deduplicated (no id), 326 ids are sent twice (6.2% of revenue counted twice, mostly within one minute: reloads), and self-referral sessions hold almost a third of purchases.

## 2. Rulebook: official sources read on 2026-11-25

In [5]:
csv_ref = pd.read_csv(Path('..') / 'data' / 'reference' / 'sources.csv')
csv_ref[~csv_ref.source.str.startswith('nager')][['source', 'http_status', 'read_on', 'keywords_found']]

,source,http_status,read_on,keywords_found
0,ga4_recommended_events,200,2026-11-25,True
1,ga4_bigquery_export_schema,200,2026-11-25,True
2,ga4_measurement_protocol,200,2026-11-25,True
3,meta_capi_deduplication,200,2026-11-25,True
4,google_tag_gateway,200,2026-11-25,True
5,merchant_center_product_data_spec,200,2026-11-25,True
6,cnil_cookies_rules,200,2026-11-25,True
7,iab_tcf_gvl_v3,200,2026-11-25,True


In [6]:
k['iab']

{'version': 186, 'updated': '2026-11-19', 'vendors': 1240, 'purposes': 11}

## 3. Tracking plan
`tracking/tracking_plan.xlsx`, `tracking/datalayer.schema.json`, GTM templates. Tested by `tests/test_datalayer.py`.

In [7]:
plan = pd.read_excel(Path('..') / 'tracking' / 'tracking_plan.xlsx')
plan.groupby('event').parameter.count().rename('parameters')

event
add_payment_info    10
add_to_cart          9
begin_checkout      10
purchase            13
view_item            9
view_item_list       8
Name: parameters, dtype: int64

## 4. Simulation (SIMULATED, Client M)

In [8]:
k['sim_rows']

{'ga4_events': 3348292,
 'orders_backend': 52768,
 'server_events': 769944,
 'pixel_events': 368493}

## 5. Reconciliation: expected vs anomalous gaps

In [9]:
csv('reconciliation_headline')

,platform,gap_when_expected,anomalous_market_weeks,unexplained_market_weeks,market_weeks
0,google_ads,0.197,7,0,88
1,meta,-0.117,14,0,88


In [10]:
csv('reconciliation_summary')[['platform', 'category', 'explanation', 'market_weeks', 'mean_ratio']]

,platform,category,explanation,market_weeks,mean_ratio
0,google_ads,anomalous,GA4 side broken: cross-domain (payment provide...,4,3.676
1,google_ads,anomalous,GA4 side broken: consent mode release (see con...,3,1.646
2,google_ads,expected,attribution model (data-driven vs last non-dir...,81,1.197
3,meta,anomalous,Meta side broken: pixel and CAPI event_id diff...,14,1.556
4,meta,expected,view-through credit and consent: within normal...,74,0.890


Google Ads stays about 23% above GA4 in the baseline weeks: attribution model, 30-day window, click date. It is explained to the client, not fixed.

## 6. Anomalies

In [11]:
csv('consent_breaks')[['market', 'dir', 'start_date', 'end_date', 'mean_deviation', 'release_at_start']]

,market,dir,start_date,end_date,mean_deviation,release_at_start
0,DE,up,2026-10-14,2026-10-27,0.193,web-v41 (New consent banner template)
1,DE,down,2026-10-28,2026-11-17,-0.334,web-v42 (Consent hotfix: block all Google tags...


In [12]:
csv('dedup_meta_breaks')

,market,first_low_day,low_days,match_rate_before,match_rate_after,release_same_day
0,ES,2026-10-07,48,1.0,0.0,web-v40 (Move Meta Purchase to the server cont...
1,FR,2026-10-07,48,1.0,0.0,web-v40 (Move Meta Purchase to the server cont...


In [13]:
csv('utm_unassigned_summary')

,agency,paid_sessions,pct_unassigned
0,AGY1,568557,0.0
1,AGY2,479455,46.1
2,AGY3,150069,5.2
3,ALL,1198081,19.1


In [14]:
csv('dedup_ga4_market').head(3)

,market,purchase_events,unique_transactions,pct_extra_purchases,without_transaction_id
0,IT,4228,3993,5.89,0
1,DE,5237,5237,0.00,0
2,ES,4266,4266,0.00,0


In [15]:
csv('feed_validation_summary')

,market,products,products_with_issue,pct_at_risk,missing_required,wrong_currency
0,DE,300,0,0.0,0.0,0.0
1,ES,300,0,0.0,0.0,0.0
2,FR,300,0,0.0,0.0,0.0
3,IT,300,0,0.0,0.0,0.0
4,NL,300,0,0.0,0.0,0.0
5,PL,300,30,10.0,9.0,21.0
6,UK,300,30,10.0,9.0,21.0
7,US,300,0,0.0,0.0,0.0


In [16]:
csv('server_alerts')

,destination,first_alert_hour,last_alert_hour,alert_hours,failed_events,alert_raised_at
0,meta_capi,2026-11-05 02:00:00,2026-11-05 08:00:00,7,582.0,2026-11-05 03:00:00


## 7. Monitoring

In [17]:
csv('monitoring_report').status.value_counts()

status
PASS    10079
FAIL      357
WARN      171
Name: count, dtype: int64

In [18]:
p = OUT / 'detection_summary.csv'
csv('detection_summary')[['event', 'name', 'check', 'first_fail', 'detection_delay', 'found']] if p.exists() else 'answer key not available'

,event,name,check,first_fail,detection_delay,found
0,1,consent_mode_misdeployed,pre_consent_hits + sessions_vs_baseline,2026-10-14,1 days 00:00:00,True
1,2,meta_double_counting,meta_event_id_match,2026-10-07,1 days 00:00:00,True
2,3,utm_non_compliance,paid_unassigned_share,2026-08-31,NaN,True
3,4,duplicate_purchases,ga4_duplicate_purchases,2026-09-01,1 days 00:00:00,True
4,5,cross_domain_broken,purchase_self_referral,2026-10-28,1 days 00:00:00,True
5,6,google_ads_vs_ga4_gap,platform_reconciliation,NaN,NaN,True
6,7,feed_currency_availability,feed_quality,2026-11-23,NaN,True
7,8,server_container_outage,server_error_rate,2026-11-05 02:00:00,0 days 01:00:00,True


In [19]:
csv('tests_results')[['name', 'passed']]

,name,passed
0,consent_breaks_dated,True
1,cross_domain_visible,True
2,demo_shop_no_external_script,True
3,duplicate_purchases_visible,True
4,feed_issues_visible,True
5,google_ads_gap_is_expected,True
6,meta_event_id_breaks,True
7,no_user_pseudo_id_in_outputs,True
8,only_placeholder_measurement_ids,True
9,real_audit_facts,True
